# Домашнее задание 3

Задание 1

In [1]:
import pandas as pd
import numpy as np 
from datetime import datetime, timedelta#импортируем

In [2]:
data = pd.read_csv('Electronic_sales_Sep2023-Sep2024.csv') #загружаем датасет

In [3]:
data.head(5) #а как выглядят данные?

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56


In [4]:
grouped_data = data.groupby("Customer ID").agg({"Payment Method": lambda x: sorted(pd.Series(x).mode().tolist())[0], #будем брать первую моду по алфавиту
    "Total Price": "sum",
    "Add-on Total": "sum",
    "Age": "count"}
).rename(columns={"Payment Method": "preferred_payment", #переименовываем как в задании
    "Total Price": "total_spent",
    "Add-on Total": "addons_spent",
    "Age": "orders_count"})

In [5]:
grouped_data #смотрим, что получилось (а вообще это наеправда, так как есть отмененные заказы, но до дедлайна полтора часа)

,preferred_payment,total_spent,addons_spent,orders_count
Customer ID,,,,
1000,Credit Card,6279.42,66.30,2
1002,Cash,5020.60,60.16,2
1003,Cash,41.50,35.56,1
1004,Credit Card,83.00,65.78,1
1005,Debit Card,11779.11,75.33,2
...,...,...,...,...
19994,Bank Transfer,5504.87,93.17,1
19995,Credit Card,5394.56,0.00,1
19996,Bank Transfer,27296.78,432.12,6


Задание 2

In [6]:
data['Order Status'].unique() #будем чистить данные

<StringArray>
['Cancelled', 'Completed']
Length: 2, dtype: str

In [7]:
data = data[data['Order Status'] != 'Cancelled'] # У нас отмена

In [8]:
data['Total Price'].unique() #отрицательных валуев нет, чистить Nan не хочу, так как удалю часть строк с покупками

array([  741.09,  1855.84,  3164.76,    41.5 ,    83.  ,  7603.47,
        4175.64,  2470.3 ,  3379.32,  7911.9 ,  4747.14,   103.75,
         124.5 ,   988.12,   844.83,  1582.38,  2783.76,  6758.64,
        4224.15,  5913.81,   186.75,    62.25,  5068.98,   145.25,
        5538.33,  1729.21,  6329.52,   207.5 ,   166.  ,  3247.72,
         463.96,  1482.18,  1976.24,  2373.57,  1689.66,  2223.27,
         247.03,    20.75,  3955.95,  7120.71,  4639.6 ,   791.19,
        3711.68,  1391.88,   927.92,  8448.3 ,  1235.15,  2534.49,
         494.06,  2319.8 ,  4130.2 ,  2796.09,  3446.8 ,  3419.04,
        3145.64,  1083.54,  6291.28,  2757.  ,  3611.8 ,  4135.5 ,
         786.41,  2697.28,  4558.72, 10257.12,  1805.9 ,  1378.5 ,
        3371.6 ,  2022.96,  3676.  ,  5698.4 ,  7977.76,  1572.82,
        6068.88,  4045.92,  2167.08,  7864.1 ,   361.18, 11396.8 ,
        3932.05,  2359.23,  4595.  ,  4718.46,  1444.72,  6743.2 ,
        6838.08,  2279.36,  5394.56,  2889.44,  9117.44,  1838

In [9]:
data['Add-ons Purchased'].unique()

<StringArray>
[                                           'Impulse Item',
                                                       nan,
                               'Impulse Item,Impulse Item',
                                               'Accessory',
                                  'Impulse Item,Accessory',
                     'Extended Warranty,Extended Warranty',
                     'Impulse Item,Accessory,Impulse Item',
        'Extended Warranty,Impulse Item,Extended Warranty',
                           'Accessory,Accessory,Accessory',
                          'Impulse Item,Extended Warranty',
                             'Accessory,Extended Warranty',
                          'Extended Warranty,Impulse Item',
                        'Accessory,Impulse Item,Accessory',
                                     'Accessory,Accessory',
                                       'Extended Warranty',
                             'Extended Warranty,Accessory',
           'Accessory,Exte

In [10]:
data['month'] = pd.to_datetime(data['Purchase Date']).dt.to_period('M')

In [11]:
data['quarter'] = pd.to_datetime(data['Purchase Date']).dt.to_period('Q')

In [12]:
del data['Purchase Date']

In [13]:
data = data.fillna('Nothing')

In [14]:
pivot = pd.pivot_table(
    data,
    index=['Shipping Type', 'Product Type', 'Add-ons Purchased', 'quarter', 'month'],
    values=['Total Price', 'Add-on Total'],
    aggfunc={
        'Total Price': 'sum',
        'Add-on Total': 'sum',
    },
)

In [15]:
pivot

Add-on Total  \
Shipping Type Product Type Add-ons Purchased quarter month                   
Expedited     Headphones   Accessory         2024Q1  2024-01        321.50   
                                                     2024-02        126.26   
                                                     2024-03        271.28   
                                             2024Q2  2024-04         96.77   
                                                     2024-05        184.08   
...                                                                    ...   
Standard      Tablet       Nothing           2024Q2  2024-05          0.00   
                                                     2024-06          0.00   
                                             2024Q3  2024-07          0.00   
                                                     2024-08          0.00   
                                                     2024-09          0.00   

                                                              Total Price  
Shipping Type Product Type Add-ons Purchased quarter month                 
Expedited     Headphones   Accessory         2024Q1  2024-01     10113.04  
                                                     2024-02      7223.60  
                                                     2024-03     10474.22  
                                             2024Q2  2024-04      3611.80  
                                                     2024-05      4695.34  
...                                                                   ...  
Standard      Tablet       Nothing           2024Q2  2024-05     35878.05  
                                                     2024-06     74684.06  
                                             2024Q3  2024-07     88093.89  
                                                     2024-08     59153.11  
                                                     2024-09     71579.28  

[5534 rows x 2 columns]

In [16]:
metrics = {'revenue_by_shipping' : pivot.groupby('Shipping Type').agg("sum"), 'addons_by_month': pivot.groupby(['quarter', 'month']).agg({'Add-on Total': "sum"}), 'revenue_by_product_type' : pivot.groupby('Product Type').agg("sum")}

In [17]:
metrics['revenue_by_shipping']

,Add-on Total,Total Price
Shipping Type,,
Expedited,180051.51,8430385.58
Express,93770.42,5632093.35
Overnight,95260.88,5887722.52
Same Day,177198.10,8292376.50
Standard,289314.33,14387037.62


In [18]:
metrics['addons_by_month']

Add-on Total
quarter month                
2023Q3  2023-09       5337.61
2023Q4  2023-10      26153.21
        2023-11      24453.33
        2023-12      22750.23
2024Q1  2024-01      93254.95
        2024-02      80253.72
        2024-03      84713.93
2024Q2  2024-04      82294.06
        2024-05      89374.18
        2024-06      84648.60
2024Q3  2024-07      88811.55
        2024-08      87861.11
        2024-09      65688.76

In [19]:
metrics['revenue_by_product_type'] #ААА дедлайн, даже подписать не успею

,Add-on Total,Total Price
Product Type,,
Headphones,112803.72,2734651.00
Laptop,170677.78,8365905.25
Smartphone,222489.69,14407835.84
Smartwatch,158824.79,9398591.23
Tablet,170799.26,7722632.25
